# Kalshi 15-minute strategy experiment

Executed companion analysis for the isolated paper ledgers and strategy-lab wallets. Results are descriptive, fee-aware, and do not treat hindsight quotes as achievable fills.

## TL;DR

- The final main paper wallet closed 6 trades at 100.0% win rate and $+14.72 net P&L.
- Across 22 matched lab markets, adaptive exit captured 90.5% of winning-market hold upside; fixed exit captured 27.1%.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'research' / 'final_experiment_analysis.py').exists():
    raise RuntimeError('Execute this notebook from the repository root')
sys.path.insert(0, str(ROOT))
from research.final_experiment_analysis import (
    ExperimentRun, analyze_experiment
)
RUN_SPECS = [('legacy_2h', 'data/simulations/20260824_all_assets_2h'), ('checkpoint', 'data/simulations/20260824_all_assets_new_models_to_6pm_v2'), ('final', 'data/simulations/20260824_final_2h_new_models_adaptive_exit')]
runs = [ExperimentRun(label, ROOT / directory) for label, directory in RUN_SPECS]
analysis = analyze_experiment(runs, ROOT / 'models/commodity_directional.json')
saved = json.loads((ROOT / 'reports/20260824_simulation_analysis/analysis_summary.json').read_text())
assert analysis['aggregate'] == saved['aggregate']
print('Independent recomputation matches saved aggregate metrics.')

Independent recomputation matches saved aggregate metrics.


## Context and methods

Main-paper trades are assigned to a run by its recorded UTC start/end bounds. Strategy-lab wallets are isolated SQLite files. Fixed and adaptive exits are compared only when a hold-to-settlement wallet entered the same ticker and side in the same session. Winning-market upside capture and losing-market loss avoidance are reported separately.

In [2]:
run_rows = []
for run in analysis['runs']:
    run_rows.append({
        'run': run['label'], 'status': run['run']['status'],
        'snapshots': run['market']['market_snapshots'],
        'depth_rows': run['market']['orderbook_levels'],
        'main_trades': run['main']['closed'],
        'main_win_rate': run['main']['win_rate'],
        'main_pnl': run['main']['net_pnl'],
        'quality_passed': run['quality']['passed'],
        'model_hash': run['run']['model_hash'],
        'config_hash': run['run']['config_hash'],
    })
pd.DataFrame(run_rows)

,run,status,snapshots,depth_rows,main_trades,main_win_rate,main_pnl,quality_passed,model_hash,config_hash
0,legacy_2h,COMPLETED,8385,313520,16,0.875,-0.62049,True,builtin:c567b079a61a622433d9179fe81d7e144d3d98...,024f73c03ca4162f6e7ff3034ad096357d7ae9b924d9da...
1,checkpoint,COMPLETED,4557,171522,2,1.000,2.11430,True,bundle:5036382a936772fc01e50e9a36a2fd886ffe91e...,97162d312603154a8f7546c09f1576cf35829d86117b11...
2,final,COMPLETED,8365,310118,6,1.000,14.71620,True,bundle:5036382a936772fc01e50e9a36a2fd886ffe91e...,60bf2fa6eef998012a1deba2f9b7153ff5de9138322923...


## Decision funnel and refusal metadata

In [3]:
gate_rows = []
for run in analysis['runs']:
    total = run['decisions']['evaluations']
    for gate, count in run['decisions'].get('gate_counts', {}).items():
        gate_rows.append({
            'run': run['label'], 'gate': gate, 'evaluations': count,
            'share': count / total if total else None,
            'candidate_yes': run['decisions'].get('candidate_outcomes', {}).get('yes', 0),
            'candidate_no': run['decisions'].get('candidate_outcomes', {}).get('no', 0),
        })
pd.DataFrame(gate_rows).sort_values(['run', 'evaluations'], ascending=[True, False])

,run,gate,evaluations,share,candidate_yes,candidate_no
14,checkpoint,entry_window,3585,0.786702,252,397
15,checkpoint,min_probability,360,0.078999,252,397
16,checkpoint,model_market_divergence,243,0.053325,252,397
17,checkpoint,candle_freshness,227,0.049813,252,397
18,checkpoint,minimum_time_to_close,96,0.021066,252,397
19,checkpoint,min_edge,31,0.006803,252,397
20,checkpoint,settlement_margin,15,0.003292,252,397
21,final,entry_window,6637,0.793425,481,943
22,final,model_market_divergence,699,0.083562,481,943
23,final,min_probability,662,0.079139,481,943


## Strategy results

In [4]:
strategy = pd.DataFrame(analysis['aggregate']['strategy_lab'])
strategy.sort_values(['net_pnl', 'closed'], ascending=[False, False])

,strategy,runs_present,closed,wins,losses,win_rate,net_pnl,pnl_per_closed_position
3,consensus_tail_carry,2,15,15,0,1.000000,7.2592,0.483947
2,complementary_pair_arb,3,0,0,0,NaN,0.0000,NaN
12,trend_trailing,3,1,0,1,0.000000,-1.7990,-1.799000
4,displaced_shelf,3,6,2,4,0.333333,-14.3224,-2.387067
1,change_point_flip,3,15,7,8,0.466667,-25.0643,-1.670953
11,strike_distance,3,21,7,14,0.333333,-30.2725,-1.441548
6,hold_to_settlement,3,57,31,26,0.543860,-34.7918,-0.610382
8,probability_acceleration,3,34,12,22,0.352941,-46.2171,-1.359326
0,adaptive_take_profit,1,22,9,13,0.409091,-51.0482,-2.320373
5,fixed_take_profit,3,68,45,23,0.661765,-60.1014,-0.883844


## Matched exit-policy comparison

In [5]:
exit_rows = []
for run in analysis['runs']:
    for row in run['matched_exit_comparison']:
        exit_rows.append({'run': run['label'], **row})
pd.DataFrame(exit_rows)

,run,strategy,matched_positions,actual_pnl,hold_counterfactual_pnl,delta_vs_hold,winner_markets,winner_upside_capture,winner_pnl,winner_hold_pnl,loser_markets,loser_exit_pnl,loser_hold_pnl,loss_avoided
0,legacy_2h,fixed_take_profit,26,-3.1227,-4.4446,1.3219,16,0.511284,6.9032,13.5017,10,-10.0259,-17.9463,7.9204
1,legacy_2h,adaptive_take_profit,0,0.0000,0.0000,0.0000,0,NaN,0.0000,0.0000,0,0.0000,0.0000,0.0000
2,checkpoint,fixed_take_profit,16,-0.8757,20.3921,-21.2678,11,0.329067,20.1778,61.3183,5,-21.0535,-40.9262,19.8727
3,checkpoint,adaptive_take_profit,0,0.0000,0.0000,0.0000,0,NaN,0.0000,0.0000,0,0.0000,0.0000,0.0000
4,final,fixed_take_profit,26,-56.1030,-12.6276,-43.4754,13,0.271187,21.5068,79.3062,13,-77.6098,-91.9338,14.3240
5,final,adaptive_take_profit,22,-51.0482,-46.7333,-4.3149,9,0.904539,40.8856,45.2005,13,-91.9338,-91.9338,0.0000


## Main-paper trade decisions

In [6]:
detail = []
for run in analysis['runs']:
    detail.extend({'run': run['label'], **row} for row in run['main_trade_details'])
pd.DataFrame(detail)

,run,trade_id,asset,ticker,outcome,status,exchange_status,placed_at,exited_at,contracts,entry_price,exit_price,entry_probability,pnl,maximum_pnl_if_winner,positive_exit_capture_ratio
0,legacy_2h,90f380dc-f4c7-4b25-b6d4-256fbe66fdc0,ETH,KXETH15M-26AUG241415-15,yes,EXITED,take_profit,2026-08-24T18:05:43.646869+00:00,2026-08-24T18:08:33.841394+00:00,2.00,0.85,0.923,0.917078,0.11600,0.28,0.414286
1,legacy_2h,03202334-d5ff-46cd-8ff6-fcfad78db333,BTC,KXBTC15M-26AUG241415-15,yes,EXITED,take_profit,2026-08-24T18:06:03.164339+00:00,2026-08-24T18:10:29.890689+00:00,2.00,0.83,0.910,0.876705,0.12840,0.32,0.401250
2,legacy_2h,20c8940d-3060-4a4d-b275-53722bc1f72f,GOLD,KXGOLD15M-26AUG241415-15,no,EXITED,take_profit,2026-08-24T18:08:43.595266+00:00,2026-08-24T18:08:52.200157+00:00,2.00,0.73,0.880,0.818091,0.25520,0.51,0.500392
3,legacy_2h,aad8e827-66f5-4f8c-afc4-4383427b6127,WTI,KXWTI15M-26AUG241500-00,yes,EXITED,take_profit,2026-08-24T18:56:08.561132+00:00,2026-08-24T18:59:18.321216+00:00,2.00,0.77,0.890,0.874999,0.19620,0.43,0.456279
4,legacy_2h,bf94c5ad-53d0-4961-a7bb-444de83cb88b,GOLD,KXGOLD15M-26AUG241500-00,yes,EXITED,take_profit,2026-08-24T18:57:58.319994+00:00,2026-08-24T18:58:08.367923+00:00,2.00,0.72,0.810,0.767624,0.12840,0.53,0.242264
5,legacy_2h,924694e4-f8c7-4bdf-9737-3c5999627c73,GOLD,KXGOLD15M-26AUG241515-15,yes,LOST,finalized,2026-08-24T19:01:38.464539+00:00,NaN,2.00,0.79,NaN,0.827491,-1.61000,0.39,NaN
6,legacy_2h,90f201a3-1ea8-45b0-add7-b8367bc8afdf,SILVER,KXSILVER15M-26AUG241515-15,yes,LOST,determined,2026-08-24T19:01:58.482534+00:00,NaN,2.00,0.76,NaN,0.805452,-1.55000,0.45,NaN
7,legacy_2h,7a4f4fac-ace5-460c-acfa-a8375a550854,BTC,KXBTC15M-26AUG241515-15,no,EXITED,take_profit,2026-08-24T19:06:48.658038+00:00,2026-08-24T19:07:18.457344+00:00,2.00,0.70,0.790,0.815238,0.12660,0.57,0.222105
8,legacy_2h,01005c2f-7e0f-4884-a291-a4879752672a,BTC,KXBTC15M-26AUG241515-15,yes,EXITED,take_profit,2026-08-24T19:12:18.342553+00:00,2026-08-24T19:12:28.323535+00:00,2.00,0.83,0.950,0.889798,0.21320,0.32,0.666250
9,legacy_2h,39a21d45-4540-4c2e-960c-4b2136adf9a9,SILVER,KXSILVER15M-26AUG241515-15,no,EXITED,take_profit,2026-08-24T19:13:57.917912+00:00,2026-08-24T19:14:06.818675+00:00,3.26,0.59,0.680,0.670016,0.18690,1.28,0.146016


## Model diagnostics

In [7]:
models = pd.DataFrame(analysis['models'])
models.sort_values('auc', ascending=False)

,asset,auc,baseline_auc,indicator_delta_auc,deployment_status,weight
3,HYPE,0.559809,0.513889,0.045920,active,0.089714
6,SOL,0.532485,0.520224,0.012262,shadow_only,0.000000
5,SILVER,0.526142,0.521152,0.004991,shadow_only,0.000000
1,DOGE,0.523349,0.491560,0.031789,shadow_only,0.000000
7,WTI,0.520297,0.493425,0.026872,shadow_only,0.000000
2,GOLD,0.512484,0.550796,-0.038312,shadow_only,0.000000
9,ZEC,0.508488,0.521338,-0.012850,shadow_only,0.000000
4,NEAR,0.507041,0.492761,0.014280,shadow_only,0.000000
8,XRP,0.497939,0.487770,0.010169,shadow_only,0.000000
0,BNB,0.464392,0.435444,0.028949,shadow_only,0.000000


## Limitations and takeaways

- The runs cover different market windows and configuration hashes; cross-run differences are descriptive, not causal. The same-window strategy-lab comparison is the strongest available evidence.
- A high win rate can coexist with negative P&L when losses dominate many small wins. Promotion therefore requires positive fee-adjusted expectancy, adequate sample size, and stable results across independent windows.
- Model AUCs near 0.50 imply weak ranking power. Only artifacts that cleared the deployment threshold were allowed a non-zero blend weight.
- No result guarantees live profitability; fill latency, queue position, and partial fills remain material execution risks.